# Chapter 14: CAPM and asset pricing

The default route computes four constructed observations with the shared asset-pricing core, checks betas 0.5 and 1.5, and draws two new constructed CAPM figures. The assumed expected risk-free and market returns are 2% and 8%; these are different objects from realized sample means. French monthly regressions remain disabled. Their historical 2010–2019 window is descriptive, without PIT vintages or trading forecasts.

[Entry modes and preparation](../NOTEBOOK_GUIDE.md). See this guide for run steps, data requirements, and expected outputs.

In [ ]:
from pathlib import Path
import sys, os, uuid
ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "pyproject.toml").is_file() and (p / "src/quant_course").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Start inside the reader package tree")
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "code"))
os.environ["MPLBACKEND"] = "Agg"
RUN_DIR = ROOT / "runs" / ("ch14_asset_pricing_" + uuid.uuid4().hex)
RUN_DIR.mkdir(parents=True, exist_ok=False)
os.chdir(ROOT)
# French branches remain disabled until data rights and execution authorization are resolved.
RUN_FRENCH = False
print("CONSTRUCTED teaching route; French market branches SKIPPED; outputs:", RUN_DIR)

from pathlib import Path
import numpy as np
import importlib.util
import sys


from quant_course.asset_pricing import (
    beta_from_covariance,
    capm_expected_return,
    constructed_two_asset_sample,
    fit_capm_excess,
)

# 复用 CLI 中经过 manifest SHA 和连续月频校验的唯一数据 loader。
_spec = importlib.util.spec_from_file_location(
    "ch14_asset_pricing_cli", ROOT / "code/ch14_asset_pricing.py"
)
ch14 = importlib.util.module_from_spec(_spec)
sys.modules[_spec.name] = ch14
_spec.loader.exec_module(ch14)


## Constructed means, covariance, beta and residuals

Give each of the four rows equal observation weight. Sample covariance uses n−1. Columns are market excess return, Asset A excess return and Asset B excess return. Subtract the matching risk-free return from asset total returns once; Mkt-RF is already excess.

In [ ]:
sample = constructed_two_asset_sample()
returns = np.column_stack([
    sample["Mkt-RF"],
    sample["Asset_A"] - sample["RF"],
    sample["Asset_B"] - sample["RF"],
])
mean_vector = returns.mean(axis=0)
covariance = np.cov(returns, rowvar=False, ddof=1)
fit_a = fit_capm_excess(returns[:, 1], returns[:, 0])
fit_b = fit_capm_excess(returns[:, 2], returns[:, 0])

print("rows are constructed; returns are decimals per single period")
print("mean [Mkt-RF, A-RF, B-RF]:", mean_vector)
print("sample covariance, ddof=1:\n", covariance)
print("beta A / B:", fit_a.beta, fit_b.beta)
print("residuals A:", fit_a.residuals)
print("residuals B:", fit_b.residuals)
assert abs(fit_a.beta - 0.5) < 1e-12
assert abs(fit_b.beta - 1.5) < 1e-12


In [ ]:
# 在另一组显式构造的期望输入下，SML 给出的模型期望收益。
rf_expected = 0.02
market_expected = 0.08
for beta in (0.5, 1.5):
    print(beta, capm_expected_return(rf_expected, beta, market_expected))

ch14.make_sml_cml_figure(RUN_DIR / "ch14_sml_cml.png")
ch14.make_beta_scatter_figure(RUN_DIR / "ch14_beta_scatter.png")
from IPython.display import Image, display
display(Image(filename=str(RUN_DIR / "ch14_sml_cml.png")))
display(Image(filename=str(RUN_DIR / "ch14_beta_scatter.png")))


The curves show model relations for assumed inputs; the points show realized means of four constructed rows. A difference neither rejects CAPM empirically nor identifies an investment opportunity.

## Disabled French NoDur regression

The optional historical loader verifies the processed SHA-256, aligns NoDur, Mkt-RF and RF, and requires 120 consecutive monthly periods from January 2010 through December 2019. HAC maxlags=5 counts monthly observations, not seconds. No market files are bundled or read by default.

In [ ]:
if RUN_FRENCH:
    monthly, industry_manifest, factor_manifest = ch14.load_french_sample()
    report, monthly_fit = ch14.french_report(monthly, industry_manifest, factor_manifest)
    print({key: report[key] for key in (
        "sample_start", "sample_end", "frequency", "n_months",
        "industry_processed_sha256", "factor_processed_sha256",
        "alpha_monthly", "alpha_hac_se_monthly", "alpha_hac_95ci_monthly",
        "beta", "r_squared", "hac", "scope",
    )})
    assert len(monthly) == 120
    assert str(monthly.index[0]) == "2010-01"
    assert str(monthly.index[-1]) == "2019-12"
    assert monthly.index.freqstr == "M"
else:
    print("SKIPPED French branch: permissions unknown; no bundled data; historical empirical results not reproduced")


In [ ]:
if RUN_FRENCH:
    ch14.make_french_regression_figure(
        monthly, monthly_fit, RUN_DIR / "ch14_nodur_regression_residuals.png"
    )
    display(Image(filename=str(RUN_DIR / "ch14_nodur_regression_residuals.png")))
else:
    print("SKIPPED French branch: permissions unknown; no bundled data; historical empirical results not reproduced")


Alpha is a monthly OLS intercept for a chosen window, market proxy and model. It is neither a direct observation of population expected alpha nor next-month or net trading return. An interval crossing zero does not establish CAPM; one excluding zero does not establish implementable arbitrage. Historical data lack monthly PIT vintages.